In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

In [4]:
train_df = pd.read_csv("/content/train.csv")

Setup (Run Before Attempting Questions) :

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
DEBERTA_MODEL = "microsoft/deberta-v3-small"
ROBERTA_MODEL = "FacebookAI/roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL, num_labels = 5)

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL, num_labels = 5)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

*
(answer format : eg - A, probability of A)

In [86]:
label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

row = train_df.iloc[25]
prompt = row["prompt"]
options = [row[option] for option in label_map.values()]

texts = [
    prompt + " [SEP] " + option
    for option in options
]

print(len(texts))
print(texts)

5
["Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [

In [23]:
deberta_model.eval()

inputs = deberta_tokenizer(
    text,
    padding = True,
    truncation = True,
    max_length = 256,
    return_tensors="pt"
)

with torch.no_grad():
  outputs = deberta_model(**inputs)

logits = outputs.logits

probs_deberta = F.softmax(logits, dim=-1)

print(probs_deberta)

predicted_label = torch.argmax(probs_deberta)

tensor([[0.1812, 0.1941, 0.1924, 0.2335, 0.1989]], dtype=torch.float16)


In [30]:
print(f"{label_map[predicted_label.item()]}, {probs_deberta[0][predicted_label.item()]:.4f}")

D, 0.2335


In [27]:
roberta_model.eval()

inputs = roberta_tokenizer(
    text,
    padding = True,
    max_length = 256,
    truncation = True,
    return_tensors = "pt"
)

with torch.no_grad():
  outputs = roberta_model(**inputs)

logits = outputs.logits

probs_roberta = F.softmax(logits, dim = -1)

print(probs_roberta)

predicted_label = torch.argmax(probs_roberta)

tensor([[0.1857, 0.2028, 0.1796, 0.2244, 0.2075]])


In [31]:
print(f"{label_map[predicted_label.item()]}, {probs_roberta[0][predicted_label.item()]:.4f}")

D, 0.2244


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [38]:
average_prob = (probs_deberta + probs_roberta)/2
print(average_prob)

predicted_label = torch.argmax(average_prob).item()

print(f"{label_map[predicted_label]}, {average_prob[0][predicted_label]:.5f}")

tensor([[0.1834, 0.1984, 0.1860, 0.2290, 0.2032]])
D, 0.22898


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [40]:
weighted_probs = 0.70*probs_deberta + 0.30*probs_roberta
print(weighted_probs)

predicted_label = torch.argmax(weighted_probs).item()

print(f"{label_map[predicted_label]}, {weighted_probs[0][predicted_label]:.5f}")

tensor([[0.1825, 0.1967, 0.1885, 0.2308, 0.2014]])
D, 0.23079


In [57]:
ranked_probs = weighted_probs.sort(descending=True)
print(ranked_probs)

top3 = ""

for item in ranked_probs[1][0]:
  top3 += label_map[item.item()]
  if(len(top3) == 3):
    break

print(f"Top 3 Prediction string of row index 25 is:\n{top3[0]} {top3[1]} {top3[2]}")

torch.return_types.sort(
values=tensor([[0.2308, 0.2014, 0.1967, 0.1885, 0.1825]]),
indices=tensor([[3, 4, 1, 2, 0]]))
Top 3 Prediction string of row index 25 is:
D E B


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [59]:
test_df = pd.read_csv("/content/test.csv")

In [83]:
from tqdm.notebook import tqdm

predictions = []

for index,row in tqdm(test_df.iterrows(),total=len(test_df), desc="Running Inference:"):
  prompt = row["prompt"]

  text = (
      f"{prompt} "
      f"[SEP] A. {row['A']} "
      f"[SEP] B. {row['B']} "
      f"[SEP] C. {row['C']} "
      f"[SEP] D. {row['D']} "
      f"[SEP] E. {row['E']}"
  )
  print(text)

  input_roberta = roberta_tokenizer(
      text,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  input_deberta = deberta_tokenizer(
      text,
      truncation = True,
      padding = True,
      max_length = 256,
      return_tensors = "pt"
  )

  with torch.no_grad():
    roberta_output = roberta_model(**input_roberta)
    deberta_output = deberta_model(**input_deberta)

  roberta_probs = F.softmax(roberta_output.logits, dim=-1)
  deberta_probs = F.softmax(deberta_output.logits, dim=-1)

  weighted_probs = 0.7*deberta_probs + 0.3*roberta_probs
  print(weighted_probs)
  top3 = ""
  weighted_probs_sorted = weighted_probs.sort(descending = True)

  for item in weighted_probs_sorted[1][0]:
    top3 += label_map[item.item()]
    if(len(top3) == 3):
      break

  prediction = f"{top3[0]} {top3[1]} {top3[2]}"
  print(prediction)

  predictions.append(prediction)
  if index ==2:
    break

Running Inference::   0%|          | 0/500 [00:00<?, ?it/s]

Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] A. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. [SEP] B. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a higher energy. [SEP] C. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a different spin. [SEP] D. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a different energy. [SEP] E. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a lower energy.
tensor([[0.1823, 0.1967, 0.1887, 0.2309, 0.2016]])
D E B
What is the estimated redshift of CEERS-93316, a candidate high-redshift galaxy observed by the James Webb Space Telescope? [SEP] A. Approximately z = 6.0, corr

In [68]:
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission_df.head()

,index,prediction
0,0,A B C
1,1,A B C
2,2,A B C
3,3,A B C
4,4,A B C
...,...,...
495,495,A B C
496,496,A B C
497,497,A B C
498,498,A B C
